# 7 · Mixed problems — the saddle point 🐎

Welcome **into the saddle**. Part II opens with its mathematical namesake: the
**saddle-point problem**. It appears whenever a minimisation is subject to a
**constraint** — the constraint is enforced by a **Lagrange multiplier**, and the
pair (solution, multiplier) is a *saddle point* of the Lagrangian, not a minimum.

The classic example is **Stokes flow** — slow, viscous, incompressible fluid:
$$ -\Delta \mathbf u + \nabla p = \mathbf f,\qquad \operatorname{div}\mathbf u = 0 . $$
The velocity $\mathbf u$ wants to minimise the viscous energy, but is **constrained**
to be divergence-free (incompressible); the **pressure** $p$ is exactly the
multiplier enforcing that constraint.

In [ ]:
# --- Google Colab: install NGSolve on first run (a no-op anywhere else) -------
# NGSolve ships its PyPI wheels as pre-releases, so the `--pre` flag is essential.
import sys
if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "--pre",
                    "ngsolve", "webgui_jupyter_widgets"], check=True)

In [ ]:
from ngsolve import *
from ngsolve import solvers
from netgen.occ import unit_square
from ngsolve.webgui import Draw

mesh = Mesh(unit_square.GenerateMesh(maxh=0.06))

## 1. The whole system at once — one product space

Discretise velocity and pressure **together** in a **product space** $X=V\times Q$ — the
**Taylor–Hood** pair, vector $H^1$ of order $k$ for $\mathbf u$ and scalar $H^1$ of order
$k-1$ for $p$ (this pairing is *inf-sup stable*: it does not lock). The whole Stokes weak
form is then **one** bilinear form on $X$,
$$ \int \nabla\mathbf u\!:\!\nabla\mathbf v + \operatorname{div}\mathbf v\,p
   + \operatorname{div}\mathbf u\,q \;=\; \int\mathbf f\!\cdot\!\mathbf v, $$
assembling into a block matrix with a tell-tale **zero pressure block**,
$K=\bigl(\begin{smallmatrix}A&B^\top\\B&0\end{smallmatrix}\bigr)$ — the signature of a
saddle point. We would love to solve it with our trusty direct `sparsecholesky`, but a
Cholesky factorisation hits a **zero pivot** on that zero block. The fix that keeps a
*direct* solve: a tiny **regularisation** $-\varepsilon\!\int p\,q$ in that block. It fills
the zero diagonal so the factorisation goes through, while barely perturbing the solution.

In [ ]:
V = VectorH1(mesh, order=2, dirichlet="bottom|right|top|left")   # velocity
Q = H1(mesh, order=1)                                            # pressure
X = V * Q                                                        # the product space
(u, p), (v, q) = X.TnT()

eps = 1e-8
a = BilinearForm(X)
a += (InnerProduct(Grad(u), Grad(v)) + div(u) * q + div(v) * p - eps * p * q) * dx
a.Assemble()
print(f"product space: {X.ndof} dofs  ({V.ndof} velocity + {Q.ndof} pressure)")

## 2. Drive it — a lid-driven cavity, solved in one shot

The square box is closed; we drag the **lid** (top edge) tangentially, with a profile
fading to zero at the corners. We set the boundary velocity, move its effect to the
right-hand side (the lifting of unit 5), and solve the **whole** system at once.

In [ ]:
gf = GridFunction(X)
lid = CF((16 * x * x * (1 - x) * (1 - x), 0))          # 1 in the middle, 0 at the corners
gf.components[0].Set(lid, definedon=mesh.Boundaries("top"))
res = (-a.mat * gf.vec).Evaluate()                     # residual carries the lid data
gf.vec.data += a.mat.Inverse(X.FreeDofs(), inverse="sparsecholesky") * res
velocity, pressure = gf.components
print(f"one direct solve done;  ‖div u‖ = {sqrt(Integrate(div(velocity)**2, mesh)):.1e}")

## 3. The pressure is fixed only up to a constant

With velocity prescribed on the *entire* boundary, the pressure appears only through its
gradient — so $p$ and $p+\text{const}$ solve the same Stokes problem. **Something** must
pin that constant. Here the **regularisation already did it** (the $-\varepsilon\!\int pq$
term makes $p$ unique, and in fact lands it close to zero mean); to get the canonical
**zero-mean** pressure exactly, subtract its average.

In [ ]:
pressure.Set(pressure - Integrate(pressure, mesh))     # canonical zero-mean pressure
print(f"∫ p dx = {Integrate(pressure, mesh):.1e}")

## 4. The flow and its pressure

The lid drags the fluid into one big **recirculating vortex**; the pressure multiplier
adjusts everywhere to keep the flow divergence-free.

In [ ]:
Draw(velocity, mesh, "velocity", vectors={"grid_size": 26})

In [ ]:
Draw(pressure, mesh, "pressure p (the constraint's multiplier)")

## 5. Refinement — the scalable block solver (`MinRes`)

A direct factorisation of the *whole* indefinite system is fine here but gets expensive in
3D. The scalable alternative keeps the blocks **separate**: assemble $K$ as a `BlockMatrix`
and solve with **`MinRes`** (the minimal-residual Krylov method, the right one for a
**symmetric indefinite** system — `CG` would misbehave), preconditioned **block-diagonally**
by the two pieces that *are* SPD: the velocity stiffness $A^{-1}$ and the **pressure mass
matrix** $M_p^{-1}$ (spectrally equivalent to the Schur complement). Each block inverse is a
cheap `sparsecholesky` — no indefinite direct solve, no regularisation needed.

In [ ]:
u_, v_ = V.TnT(); p_, q_ = Q.TnT()
Amat = BilinearForm(InnerProduct(Grad(u_), Grad(v_)) * dx).Assemble()
Bmat = BilinearForm(trialspace=V, testspace=Q); Bmat += div(u_) * q_ * dx; Bmat.Assemble()
Mp   = BilinearForm(p_ * q_ * dx).Assemble()                          # pressure mass matrix
K = BlockMatrix([[Amat.mat, Bmat.mat.T], [Bmat.mat, None]])           # the saddle operator
C = BlockMatrix([[Amat.mat.Inverse(V.FreeDofs(), inverse="sparsecholesky"), None],
                 [None, Mp.mat.Inverse(inverse="sparsecholesky")]])   # block preconditioner

gu = GridFunction(V); gp = GridFunction(Q)
gu.Set(lid, definedon=mesh.Boundaries("top"))
rhs = BlockVector([(-Amat.mat * gu.vec).Evaluate(), (-Bmat.mat * gu.vec).Evaluate()])
du = gu.vec.CreateVector(); du[:] = 0
dp = gp.vec.CreateVector(); dp[:] = 0
with TaskManager():
    solvers.MinRes(mat=K, pre=C, rhs=rhs, sol=BlockVector([du, dp]),
                   maxsteps=500, tol=1e-10, printrates=False)
gu.vec.data += du; gp.vec.data += dp
gp.Set(gp - Integrate(gp, mesh))
print(f"block MinRes done;  ‖div u‖ = {sqrt(Integrate(div(gu)**2, mesh)):.1e}  "
      f"(same flow, scalable solver)")

This splitting idea — a hard system tamed either by a direct solve with a nudge, or by
inverting only the pieces you *can* — builds on the **solver toolbox** (notebook 6). Next,
we stay in the saddle and learn to label **materials and boundaries**.

In [ ]:
# Navigation between units — shown only in a live notebook (Colab / JupyterLite /
# local Jupyter), never in the rendered website (which has its own prev/next nav).
import os, sys
if not os.environ.get("WEBGUI_SCENE_DIR"):          # not the static site build
    _prev = ("06-linear-solvers", "6 · The solver toolbox 🛠")
    _next = ("08-unsteady-doubleglazing", "8 · Unsteady problems — the double-glazing flow")
    def _u(_nb):
        if "google.colab" in sys.modules:
            return "https://colab.research.google.com/github/schruste/ngsum2026-colab/blob/colab/" + _nb + ".ipynb"
        return _nb + ".ipynb"                       # JupyterLite & local: relative .ipynb link
    _parts  = ["⬅️ **Previous:** [%s](%s)" % (_prev[1], _u(_prev[0]))] if _prev else []
    _parts += ["➡️ **Next:** [%s](%s)" % (_next[1], _u(_next[0]))] if _next else []
    from IPython.display import display, Markdown
    display(Markdown(" · ".join(_parts)))